# NB04 — Baselines (5-class, test-20%)

Non-transformer baselines on `label5`, all reported on the **same pure 20% test** as the main models:
1. **TF-IDF (word) + Logistic Regression**
2. **TF-IDF (word+char) + Linear SVM**
3. **Word-embedding BiLSTM** (neural, non-transformer)

Classical models train on **train+val (80%)** (no early stopping needed); the BiLSTM trains on train,
early-stops on val. All evaluated on the held-out 20% test. Metrics: macro-F1, weighted-F1, acc, MCC, AUROC.


In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (f1_score, accuracy_score, matthews_corrcoef,
                             roc_auc_score, classification_report)
warnings.filterwarnings("ignore")

SPLIT_DIR = "../data/splits"; OUT = "../outputs/baselines"; os.makedirs(OUT, exist_ok=True)
LABEL, TEXT = "label5", "text_clean"
train = pd.read_csv(f"{SPLIT_DIR}/random_train.csv")
val   = pd.read_csv(f"{SPLIT_DIR}/random_val.csv")
test  = pd.read_csv(f"{SPLIT_DIR}/random_test.csv")
classes = sorted(train[LABEL].unique()); enc = {c:i for i,c in enumerate(classes)}
print("classes:", enc)

tr = pd.concat([train, val], ignore_index=True)          # classical: train on 80%
Xtr_txt, ytr = tr[TEXT].fillna("").astype(str), tr[LABEL].map(enc).values
Xte_txt, yte = test[TEXT].fillna("").astype(str), test[LABEL].map(enc).values
print(f"train+val={len(tr):,}  test(20%)={len(test):,}")

def report(name, yt, yp, proba=None):
    rec = {"model": name,
           "macro_f1": round(float(f1_score(yt,yp,average='macro',zero_division=0)),4),
           "weighted_f1": round(float(f1_score(yt,yp,average='weighted',zero_division=0)),4),
           "accuracy": round(float(accuracy_score(yt,yp)),4),
           "mcc": round(float(matthews_corrcoef(yt,yp)),4)}
    if proba is not None:
        try: rec["auroc"] = round(float(roc_auc_score(yt,proba,multi_class='ovr',average='macro',
                                                       labels=list(range(len(classes))))),4)
        except Exception: rec["auroc"] = float('nan')
    else: rec["auroc"] = float('nan')
    print(f"\n{name}: macroF1={rec['macro_f1']} wF1={rec['weighted_f1']} acc={rec['accuracy']} mcc={rec['mcc']} auroc={rec['auroc']}")
    print(classification_report(yt, yp, target_names=classes, zero_division=0))
    return rec

In [ ]:
# ── 1) TF-IDF (word) + Logistic Regression ──────────────────────────────────
results = []
t0 = time.time()
vec_w = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=3, max_features=50000, sublinear_tf=True)
Xtr = vec_w.fit_transform(Xtr_txt); Xte = vec_w.transform(Xte_txt)
lr = LogisticRegression(max_iter=2000, class_weight="balanced", C=2.0, n_jobs=-1)
lr.fit(Xtr, ytr)
results.append(report("TFIDF(word)+LogReg", yte, lr.predict(Xte), lr.predict_proba(Xte)))
print(f"[{time.time()-t0:.0f}s]")

In [ ]:
# ── 2) TF-IDF (word+char) + Linear SVM ──────────────────────────────────────
t0 = time.time()
vec_c = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,5), min_df=3, max_features=100000, sublinear_tf=True)
from scipy.sparse import hstack
Xtr2 = hstack([Xtr, vec_c.fit_transform(Xtr_txt)]).tocsr()
Xte2 = hstack([Xte, vec_c.transform(Xte_txt)]).tocsr()
svm = CalibratedClassifierCV(LinearSVC(class_weight="balanced", C=1.0), cv=3)   # calibrated → proba for AUROC
svm.fit(Xtr2, ytr)
results.append(report("TFIDF(word+char)+LinearSVM", yte, svm.predict(Xte2), svm.predict_proba(Xte2)))
print(f"[{time.time()-t0:.0f}s]")

In [ ]:
# ── 3) Word-embedding BiLSTM (neural, non-transformer) ──────────────────────
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42); np.random.seed(42)

# tiny whitespace vocab from TRAIN ONLY (no leakage)
from collections import Counter
MAXLEN, MIN_FREQ, EMB, HID, EPOCHS, BS = 64, 2, 200, 256, 8, 64
cnt = Counter(tok for t in train[TEXT].fillna("").astype(str) for tok in t.split())
vocab = {"<pad>":0, "<unk>":1}
for w,c in cnt.items():
    if c >= MIN_FREQ: vocab[w] = len(vocab)
def encode_txt(s):
    ids = [vocab.get(w,1) for w in str(s).split()[:MAXLEN]]
    return ids + [0]*(MAXLEN-len(ids))
print(f"vocab={len(vocab):,}")

class TxtDS(Dataset):
    def __init__(self, df): 
        self.x=[encode_txt(t) for t in df[TEXT].fillna("").astype(str)]; self.y=df[LABEL].map(enc).values
    def __len__(self): return len(self.x)
    def __getitem__(self,i): return torch.tensor(self.x[i]), torch.tensor(int(self.y[i]))

class BiLSTM(nn.Module):
    def __init__(self, V, E, H, C):
        super().__init__(); self.emb=nn.Embedding(V,E,padding_idx=0)
        self.lstm=nn.LSTM(E,H,batch_first=True,bidirectional=True,num_layers=1)
        self.drop=nn.Dropout(0.3); self.fc=nn.Linear(2*H,C)
    def forward(self,x):
        m=(x!=0).float().unsqueeze(-1); h,_=self.lstm(self.emb(x))
        pooled=(h*m).sum(1)/m.sum(1).clamp(1); return self.fc(self.drop(pooled))

tr_loader = DataLoader(TxtDS(train), batch_size=BS, shuffle=True)
va_loader = DataLoader(TxtDS(val),  batch_size=128)
te_loader = DataLoader(TxtDS(test), batch_size=128)
cw = torch.tensor([len(ytr)/(len(classes)*max((ytr==i).sum(),1)) for i in range(len(classes))],
                  dtype=torch.float32, device=device)
model = BiLSTM(len(vocab),EMB,HID,len(classes)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss(weight=cw)

@torch.no_grad()
def predict(loader):
    model.eval(); ps=[]
    for x,_ in loader:
        ps.append(F.softmax(model(x.to(device)),-1).cpu())
    return torch.cat(ps).numpy()

best, best_state = -1, None
for ep in range(EPOCHS):
    model.train()
    for x,y in tr_loader:
        opt.zero_grad(); loss=crit(model(x.to(device)), y.to(device)); loss.backward(); opt.step()
    vp = predict(va_loader).argmax(-1); vf=f1_score(val[LABEL].map(enc).values, vp, average="macro", zero_division=0)
    if vf>best: best, best_state = vf, {k:v.cpu().clone() for k,v in model.state_dict().items()}
    print(f"  ep{ep+1} val_macroF1={vf:.4f}")
model.load_state_dict(best_state)
proba = predict(te_loader)
results.append(report("BiLSTM", yte, proba.argmax(-1), proba))

In [ ]:
# ── Summary ─────────────────────────────────────────────────────────────────
sdf = pd.DataFrame(results)[["model","macro_f1","weighted_f1","accuracy","mcc","auroc"]]
print(sdf.to_string(index=False))
sdf.to_csv(f"{OUT}/baseline_results.csv", index=False)
print(f"\n✅ saved {OUT}/baseline_results.csv")